# Week 01: Transformers

## Learning Objectives

By the end of this chapter, you should be able to:

- Understand why Transformers were introduced.
- Explain limitations of RNNs and LSTMs.
- Understand Self-Attention and its purpose.
- Differentiate between Self-Attention and Cross-Attention.
- Explain Masked Attention.
- Understand Positional Encoding.
- Understand Multi-Head Attention.
- Interpret Transformer architecture diagrams.
- Calculate attention scores manually.
- Implement simple attention mechanisms in Python.

---



## Key Concepts

### Evolution of Sequence Models

| Model | Advantage | Limitation |
|---------|---------|---------|
| RNN | Sequential processing | Vanishing gradients |
| LSTM | Better long-range memory | Slow training |
| GRU | Simpler than LSTM | Sequential bottleneck |
| Transformer | Parallel processing | High memory usage |

### Transformer Components

1. Input Embeddings
2. Positional Encoding
3. Self-Attention
4. Multi-Head Attention
5. Feed Forward Network
6. Residual Connections
7. Layer Normalization

### Types of Attention

- Self Attention
- Cross Attention
- Masked Attention

---



## Mathematical Foundation

### Attention Equation

$$
Attention(Q,K,V) = softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

Where:

- Q = Query
- K = Key
- V = Value
- d_k = Dimension of key vectors

---

### Step 1: Similarity Score

$$
Score = QK^T
$$

Measures similarity between tokens.

---

### Step 2: Scaling

$$
\frac{QK^T}{\sqrt{d_k}}
$$

Prevents extremely large values.

---

### Step 3: Softmax

$$
softmax(x_i) = \frac{e^{x_i}} {\sum_j e^{x_j}}
$$

Converts scores into probabilities.

---

### Step 4: Weighted Sum

$$
\text{Output} = \text{AttentionWeights} \times {V}
$$

Produces contextualized representation.

---

## Architecture Diagram

### Transformer Architecture

```text
      Input Tokens
          │
          ▼
      Embeddings
          │
          ▼
    Positional Encoding
          │
          ▼
    Multi Head Attention
          │
          ▼
      Add & Norm
          │
          ▼
    Feed Forward Network
          │
          ▼
      Add & Norm
          │
          ▼
        Output
```

---

### Encoder-Decoder Structure

```text
Encoder
  ├── Self Attention
  ├── Feed Forward
  └── Context Vector

Decoder
  ├── Masked Attention
  ├── Cross Attention
  ├── Feed Forward
  └── Prediction
```
---


![encoder-image](../assets/images/image01-encoderdecoder.png)

## Detailed Explanation

### Why Transformers?

Traditional RNNs process sequences one token at a time.

Example:

```text
I → love → learning → transformers
```

Each word waits for the previous word.

Problems:

* Slow training
* Difficult long-range dependencies
* Limited parallelization

Transformers solve these issues using attention mechanisms.



---

### Self Attention

Each token looks at every other token.

Sentence:

```text
The animal didn't cross the street because it was tired.
```

The model learns:

```text
it → animal
```

through attention weights.

![multihead-attension](../assets/images/image02-selfandmultiheadattension.png)
---

### Cross Attention

Used in encoder-decoder models.

Example:

```text
English:
I love AI

French:
J'aime l'IA
```

Decoder attends to encoder outputs.

---

### Masked Attention

Prevents future token leakage.

Example:

```text
I love ______
```

Prediction must not see future words.

Used in GPT.

---

### Positional Encoding

Transformers have no natural notion of order.

Positional encoding injects position information.

Example:

```text
Word  Position

I      1
love   2
AI     3
```

Sinusoidal encoding:

$PE(pos,2i) = sin\left(\frac{pos}{10000^{2i/d}}\right)$

$PE(pos,2i+1) = cos\left(\frac{pos}{10000^{2i/d}}\right)$

---

## Multi-Head Attention (MHA)

### Parameters and Dimensions

- **Sequence Length:** $(t)$
- **Number of Heads:** $(h)$
- **Embedding Dimension:** $(d_{model})$
- **Input:** $X \in \mathbb{R}^{d_{model} \times t}$
- **Head Dimensions:**
  - $d_k = d_q = \frac{d_{model}}{h}$
  - $d_v = \text{user-defined}$
- **Weight Matrices:**
  - $W_Q \in \mathbb{R}^{d_q \times d_{model}}$
  - $W_K \in \mathbb{R}^{d_k \times d_{model}}$
  - $W_V \in \mathbb{R}^{d_v \times d_{model}}$
- **If all heads have the same value dimension:**
  - $W_O \in \mathbb{R}^{d_{model} \times (h * d_v)}$
- **If each head has a different value dimension:**
  - $W_O \in \mathbb{R}^{d_{model} \times (d_{v1}+d_{v2}+\cdots+d_{vh})}$

---



## Calculations Methodology

- **Query:** $Q = \text{matmul}(W_Q,X) \in \mathbb{R}^{d_q \times t}$
- **Key:** $K = \text{matmul}(W_K,X) \in \mathbb{R}^{d_k \times t}$
- **Value:** $V = \text{matmul}(W_V,X) \in \mathbb{R}^{d_v \times t}$
- **Attention Scores:** $\text{matmul}(Q^T,K) \in \mathbb{R}^{t \times t}$
- **Scaled Dot-Product Attention:** $\text{softmax}\left(\frac{Q^TK}{\sqrt{d_k}}\right)V^T\in\mathbb{R}^{t \times d_v}$

$$
\operatorname{softmax}
\left(
\frac{Q^{T}K}{\sqrt{d_k}}
\right)
=
\operatorname{softmax}
\left(
\begin{bmatrix}
p & q\\
r & s
\end{bmatrix}
\right)
=
\begin{bmatrix}
\dfrac{e^{p}}{e^{p}+e^{q}} &
\dfrac{e^{q}}{e^{p}+e^{q}}\\[10pt]
\dfrac{e^{r}}{e^{r}+e^{s}} &
\dfrac{e^{s}}{e^{r}+e^{s}}
\end{bmatrix}
$$

- **Concatenation:** 
  - $Z = [\text{head}_1 \;\text{head}_2\cdots\text{head}_h]$

  If each head has different value dimensions:

  - $Z = [t\times d_{v1}][t\times d_{v2}]\cdots[t\times d_{vh}]\in\mathbb{R}^{t\times(d_{v1}+d_{v2}+\cdots+d_{vh})}$

  If all heads have equal value dimensions $d_v$:

  - $Z = [t\times d_v][t\times d_v]\cdots[t\times d_v]\in\mathbb{R}^{t\times(h\cdot d_v)}$
- **Multi Head Attention (MHA):**
  - $\Large MHA = [W_o]_{d_{model}\times(h*d_v)}[Z^T]_{(h*d_v)\times t} \in \mathbb{R}^{d_{model}\times t}$
---


### Example

- **Sequence Length** : $t = 2$
- **Number of Heads** : $h = 2$
- **Embedding dimension** : $d_{model} = 4$
- **Input** $X \in \mathbb{R}^{d_{model}\times t}$
  - $\Large X = \begin{bmatrix}1&0\\0&2\\1&0\\0&2\end{bmatrix}$
- $d_k = d_q = \Large \frac{d_{model}}{h} = \Large \frac{4}{2} = 2$
- $d_v = 2$
- **Weight matrices for** $W_Q, W_K, W_V$ *first head:*
  - $[Head\ 1]$ Weights: $W_Q^{(1)} = W_K^{(1)} = W_V^{(1)} = \begin{bmatrix}1&1&1&1\\2&2&2&2\end{bmatrix}\in\mathbb{R}^{d_q\times d_{model}}$
- **Weight matrices for** $W_Q, W_K, W_V$ *Second head:*
  - $[Head\ 2]$ Weights: $W_Q^{(2)} = W_K^{(2)} = W_V^{(2)} = \begin{bmatrix}1&0&1&0\\ 1&1&2&1\end{bmatrix}\in\mathbb{R}^{d_q\times d_{model}}$
- Output Projection Matrix
  - $W_O = \begin{bmatrix}0.5&0&0&0\\0&0.5&0&0\\0&0&1&0\\0&0&0&1\end{bmatrix}\in \mathbb{R}^{d_{model}\times(h*d_v)} $

---



- Self Attention Formula
  - $\text{Attention} = \text{softmax}\left(\frac{Q^TK}{\sqrt{d_k}}\right)V^T$

  ![alt text](image-1.png)

- $[Head\ 1] =>$  **Compute $Q^1, K^1, V^1$**

  $
  \begin{aligned}Q^{(1)} = K^{(1)} = V^{(1)}
  &=\begin{bmatrix}1 & 1 & 1 & 1\\2 & 2 & 2 & 2\end{bmatrix} X \\
  &= \begin{bmatrix} 1 & 1 & 1 & 1\\2 & 2 & 2 & 2\end{bmatrix}\begin{bmatrix}1 & 0\\0 & 2\\1 & 0\\0 & 2\end{bmatrix} \\
  &= \begin{bmatrix}2 & 4\\4 & 8\end{bmatrix}
  \end{aligned}
  $

- **Attention Scores:**

  $
  \begin{aligned}\frac{Q^TK}{\sqrt{2}} = \frac{1}{\sqrt2}\begin{bmatrix} 2&4\\4&8\end{bmatrix}\begin{bmatrix}2&4\\4&8\end{bmatrix}
  &= \begin{bmatrix}14.14&28.28\\28.28&56.56\end{bmatrix}\end{aligned}$

### Softmax

- $\text{softmax} = \begin{bmatrix}0&1\\0&1\end{bmatrix}$

### Head 1 Output
- $\begin{bmatrix}0&1\\0&1\end{bmatrix}\begin{bmatrix}2&4\\4&8\end{bmatrix} = \begin{bmatrix}4&8\\4&8\end{bmatrix}$

### Head 2

### Compute Q, K, V
- $Q^{(2)}, K^{(2)}, V^{(2)} = \begin{bmatrix}1&0&1&0\\1&1&2&1\end{bmatrix} X = \begin{bmatrix}2&0\\3&4\end{bmatrix}$

### Softmax
- $\text{softmax}\left(\frac{Q^TK}{\sqrt{d_k}}\right)V^T$

$\Large
\begin{bmatrix}
\frac{e^{9.19}}{e^{9.19} + e^{8.48}}&\frac{e^{9.19}}{e^{9.19} + e^{8.48}}\\
\frac{e^{8.48}}{e^{8.48} + e^{11.31}}&\frac{e^{8.48}}{e^{8.48} + e^{11.31}}
\end{bmatrix}
$ 

$= \begin{bmatrix}0.67&0.33\\0.06&0.94\end{bmatrix}$

### Head 2 Output
- $\begin{bmatrix}0.67&0.33\\0.06&0.94\end{bmatrix}\begin{bmatrix}2&0\\3&4\end{bmatrix} =\begin{bmatrix}1.34&3.33\\0.11&3.94\end{bmatrix}$

---

### Multi-Head Attention

![alt text](image.png)



**Concatenate**
- $Z =[\text{Head}_1][\text{Head}_2] = \begin{bmatrix}\begin{bmatrix}4&8\\4&8\end{bmatrix}\begin{bmatrix}1.34&3.33\\0.11&3.94\end{bmatrix}\end{bmatrix}$

**Multi Head Attention (MHA)**

$\Large MHA = [W_o]_{d_{model}\times(h*d_v)}[Z^T]_{(h*d_v)\times t} \in \mathbb{R}^{d_{model}\times t}$

$\begin{bmatrix}0.5&0&0&0\\0&0.5&0&0\\0&0&0.5&0\\0&0&0&0.5\end{bmatrix}\begin{bmatrix}4&4\\8&8\\1.34&0.11\\3.33&3.94\end{bmatrix} = \begin{bmatrix}2&2\\4&4\\1.34&0.11\\3.33&3.94\end{bmatrix}$

#### Final Output
- $\boxed{\text{MHA} = \begin{bmatrix}2&2\\4&4\\1.34&0.11\\3.33&3.94\end{bmatrix}}$

## Examples with solution

- Sentence: ```Machine Learning is powerful```
- Assume:

  ```text
  Q = [1,0]
  K = [1,1]
  V = [2,3]
  ```

- Similarity : $QK^T = 1(1)+0(1) = 1$
- Scaling : $\frac{1}{\sqrt2} = 0.707$
- Softmax : $softmax(0.707) = 1$
- Output : $1 \times [2,3] = [2,3]$
- Final attention output : ```[2,3]```

---

### Problem Statement

A dataset contains four samples:

1. I enjoyed the movie Transformers
2. We live among its people now
3. They have much to learn
4. Freedom is the right of all sentient beings

Each word embedding has dimension: $\mathbb{R}^{1 \times 256}$

The transformer processes each sequence in its entirety.

Projection matrices: $$W_Q, W_K, W_V \in \mathbb{R}^{256 \times 64}$$

Number of attention heads: $h = 4$

Output projection matrix: $$W_O \in \mathbb{R}^{256 \times 256}$$

---

### Q1. What is the minimum context length (window) T?

- Step 1: Count the number of words in each sequence
  - Sequence 1: ```I enjoyed the movie Transformers```
    - Number of words: $5$
  - Sequence 2: ```We live among its people now```
    - Number of words: $6$
  - Sequence 3: ```They have much to learn```
    - Number of words: $5$
  - Sequence 4: ```Freedom is the right of all sentient beings```
    - Number of words: $8$
- Step 2: Determine Context Length
  - The transformer must process the longest sequence completely.
  - Therefore: $T = \max(5,6,5,8)$

**Answer:** $\boxed{T = 8}$

The minimum context window required is: ```8 tokens```

---

### Q2. What is the dimension of the encoder input tensor?

- Given
  - Embedding dimension: $d_{model}=256$
  - Batch size: There are 4 sequences. $B=4$
  - Context length: $T=8$
- Transformer Input Shape
  - Transformer encoders typically accept: $(B,T,d_{model})$ where
    - B = batch size
    - T = sequence length
    - d_model = embedding dimension
- Substituting values: $(4,8,256)$

**Answer:** $\boxed{(4,8,256)}$

Input tensor shape: ```python(batch_size, sequence_length, embedding_dimension)(4, 8, 256)```

---

### Q3. Number of parameters in Multi-Head Attention

- Given
  - Number of heads: $h=4$
  - Projection matrices:
    - $W_Q \in \mathbb{R}^{256\times64}$
    - $W_K \in \mathbb{R}^{256\times64}$
    - $W_V \in \mathbb{R}^{256\times64}$
- Parameters per Head
  - For one head:
    - Query Matrix $256\times64 = 16384$
    - Key Matrix $256\times64 = 16384$
    - Value Matrix $256\times64\ = 16384$
    - Total : $16384 + 16384 + 16384 = 49152$
  - Four Attention Heads $4\times49152 = 196608$
  - Output Projection
    - Given: $W_O \in \mathbb{R}^{256\times256}$
    - Parameters: $256\times256 = 5536$
- Total Multi-Head Attention Parameters: $196608 + 65536 = 262144$

**Answer:** $\boxed{262144}$

Total parameters: ```262,144```

---

### Verification Using Python

In [ ]:
embedding_dim = 256
head_dim = 64
num_heads = 4

q_params = embedding_dim * head_dim
k_params = embedding_dim * head_dim
v_params = embedding_dim * head_dim

per_head = q_params + k_params + v_params

all_heads = num_heads * per_head

wo_params = embedding_dim * embedding_dim

total = all_heads + wo_params

print("Per Head Parameters :", per_head)
print("All Heads Parameters:", all_heads)
print("WO Parameters       :", wo_params)
print("Total Parameters    :", total)


Expected Output:

```text
Per Head Parameters : 49152
All Heads Parameters: 196608
WO Parameters       : 65536
Total Parameters    : 262144
```

---

### Key Learning

- For a transformer: $\text{Input Shape} = (B,T,d_{model})$
- Multi-head attention parameters: $h\times \left(W_Q + W_K + W_V\right) + W_O$
- Understanding these calculations is essential for estimating memory usage, model size, and computational cost in modern LLMs.

## Python Demonstration

### Basic Self-Attention

In [3]:
import numpy as np

Q = np.array([[1, 0]])
K = np.array([[1, 1]])
V = np.array([[2, 3]])

scores = Q @ K.T

scaled_scores = scores / np.sqrt(K.shape[1])

weights = np.exp(scaled_scores)
weights = weights / weights.sum()

output = weights @ V

print(output)

[[2. 3.]]


---

### Hugging Face Example

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

text = "Transformers changed NLP"

tokens = tokenizer(text)

print(tokens)

{'input_ids': [101, 19081, 2904, 17953, 2361, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1]}


## Industry Perspective

### Applications

#### ChatGPT

Uses:

* Decoder-only Transformer
* Masked Attention
* Autoregressive generation

#### BERT

Uses:

* Encoder-only Transformer
* Bidirectional attention

Applications:

* Search
* Classification
* Information Retrieval

#### T5

Uses:

* Encoder-Decoder architecture

Applications:

* Translation
* Summarization
* Question Answering

---

## Interview Questions

### Conceptual

1. Why are Transformers faster than RNNs?

2. What is Self-Attention?

3. Difference between Self-Attention and Cross-Attention?

4. Why is scaling performed in attention?

5. Why do we need Positional Encoding?

6. Why use Multi-Head Attention?

---

### Mathematical

1. Derive attention formula.

2. Explain softmax.

3. Compute attention output manually.

4. Why divide by √dk?

---

### Practical

1. GPT uses which attention?

2. BERT uses which attention?

3. Which architecture is used in machine translation?

4. Why is Transformer training expensive?

---

## Summary

### Key Takeaways

* Transformers replaced RNNs for most NLP tasks.
* Attention is the core building block.
* Self-Attention captures token relationships.
* Cross-Attention connects encoder and decoder.
* Masked Attention enables autoregressive generation.
* Positional Encoding provides sequence order information.
* Multi-Head Attention improves representation learning.
* Transformers enable parallel training and large-scale learning.

---

### References

#### Research Papers

#### Attention Is All You Need (2017)

**Authors:** Ashish Vaswani et al.

**Paper:**  https://arxiv.org/abs/1706.03762

```bibtex
@article{vaswani2017attention,
  title={Attention Is All You Need},
  author={Vaswani, Ashish and others},
  journal={Advances in Neural Information Processing Systems},
  year={2017}
}
```

---

#### BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding (2018)

**Authors:** Jacob Devlin et al.

**Paper:**  https://arxiv.org/abs/1810.04805

```bibtex
@article{devlin2018bert,
  title={BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding},
  author={Devlin, Jacob and Chang, Ming-Wei and Lee, Kenton and Toutanova, Kristina},
  year={2018}
}
```

---

#### GPT-3: Language Models are Few-Shot Learners (2020)

**Authors:** Tom B. Brown et al.

**Paper:**  https://arxiv.org/abs/2005.14165

```bibtex
@article{brown2020gpt3,
  title={Language Models are Few-Shot Learners},
  author={Brown, Tom B. and others},
  journal={Advances in Neural Information Processing Systems},
  year={2020}
}
```

---

#### T5: Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer (2020)

**Authors:** Colin Raffel et al.

**Paper:**  https://arxiv.org/abs/1910.10683

```bibtex
@article{raffel2020t5,
  title={Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer},
  author={Raffel, Colin and others},
  year={2020}
}
```

---

#### SentencePiece: A Simple and Language Independent Subword Tokenizer (2018)

**Authors:** Taku Kudo and John Richardson

**Paper:**  https://arxiv.org/abs/1808.06226

```bibtex
@article{kudo2018sentencepiece,
  title={SentencePiece: A Simple and Language Independent Subword Tokenizer and Detokenizer for Neural Text Processing},
  author={Kudo, Taku and Richardson, John},
  year={2018}
}
```

---

#### PaLM: Scaling Language Modeling with Pathways (2022)

**Authors:** Aakanksha Chowdhery et al.

**Paper:**  https://arxiv.org/abs/2204.02311

```bibtex
@article{chowdhery2022palm,
  title={PaLM: Scaling Language Modeling with Pathways},
  author={Chowdhery, Aakanksha and others},
  year={2022}
}
```

---

#### Chinchilla: Training Compute-Optimal Large Language Models (2022)

**Authors:** Jordan Hoffmann et al.

**Paper:**  https://arxiv.org/abs/2203.15556

```bibtex
@article{hoffmann2022chinchilla,
  title={Training Compute-Optimal Large Language Models},
  author={Hoffmann, Jordan and others},
  year={2022}
}
```

---

#### LoRA: Low-Rank Adaptation of Large Language Models (2021)

**Authors:** Edward J. Hu et al.

**Paper:**  https://arxiv.org/abs/2106.09685

```bibtex
@article{hu2021lora,
  title={LoRA: Low-Rank Adaptation of Large Language Models},
  author={Hu, Edward J. and others},
  year={2021}
}
```

---

#### QLoRA: Efficient Finetuning of Quantized LLMs (2023)

**Authors:** Tim Dettmers et al.

**Paper:**  https://arxiv.org/abs/2305.14314

```bibtex
@article{dettmers2023qlora,
  title={QLoRA: Efficient Finetuning of Quantized LLMs},
  author={Dettmers, Tim and others},
  year={2023}
}
```

---

#### Llama 3 Technical Report (2024)

**Authors:** Meta AI

**Paper:**  https://arxiv.org/abs/2407.21783

```bibtex
@article{dubey2024llama3,
  title={The Llama 3 Herd of Models},
  author={Dubey, Abhimanyu and others},
  year={2024}
}
```

---

### Documentation

- [PyTorch](https://pytorch.org/docs/stable/)
- [Hugging Face Transformers](https://huggingface.co/docs/transformers/)
- [Hugging Face Datasets](https://huggingface.co/docs/datasets/)
- [Hugging Face Evaluate](https://huggingface.co/docs/evaluate/)
- [Jupyter Book](https://jupyterbook.org/)
- [NumPy](https://numpy.org/doc/)
- [Pandas](https://pandas.pydata.org/docs/)
- [Scikit-Learn](https://scikit-learn.org/stable/)

---

### Further Reading

- Dive Into Deep Learning (D2L) https://d2l.ai/

  Recommended Chapters:
  - Attention Mechanisms
  - Transformers
  - BERT
  - Sequence Models


- Stanford CS224N

- Natural Language Processing with Deep Learning https://web.stanford.edu/class/cs224n/

  Recommended Lectures:
  - Word Embeddings
  - Attention
  - Transformers
  - Pretrained Language Models
  - Large Language Models

- The Illustrated Transformer https://jalammar.github.io/illustrated-transformer/

- The Illustrated GPT-2 https://jalammar.github.io/illustrated-gpt2/

- Hugging Face NLP Course https://huggingface.co/learn/nlp-course/

- Full Stack Deep Learning https://fullstackdeeplearning.com/